# Inteligencia Computacional — Guía de trabajos prácticos 2


In [ ]:
# Recarga automatica de src/: si editas un archivo de src, el notebook toma la
# version nueva sin tener que reiniciar el kernel.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path

from src.data_loader import cargar_patrones, armar_arquitectura
from src.graficos import (
    graficar_y_animar_regiones,
    graficar_comparacion,
    graficar_clasificacion,
    graficar_curvas_por_tasa,
)

DIRECTORIO_DATASET = Path("Dataset")
DIRECTORIO_GRAFICOS = Path("Graficos")


class PerceptronMulticapa:
    """Perceptron multicapa entrenado con retropropagacion.

    La cantidad de capas y de neuronas por capa se elige libremente con la lista
    `neuronas_por_capa`: [2, 4, 1] es 2 entradas, una capa oculta de 4 y 1 salida.
    """

    def __init__(self, neuronas_por_capa, tasa_aprendizaje=0.1, semilla=None):
        # Generador propio: con la misma semilla, la misma red inicial (reproducible).
        generador = np.random.default_rng(semilla)

        self.neuronas_por_capa = neuronas_por_capa
        self.tasa_aprendizaje = tasa_aprendizaje

        # Una matriz de pesos por capa de CONEXIONES. Con N niveles de neuronas hay N-1
        # juegos de cables, de ahi el "- 1". Forma (cuantas salen, cuantas llegan).
        self.pesos = [
            generador.uniform(-0.5, 0.5, (neuronas_por_capa[i], neuronas_por_capa[i + 1]))
            for i in range(len(neuronas_por_capa) - 1)
        ]

        # Un umbral por neurona QUE RECIBE, por eso son vectores y no matrices.
        self.umbrales = [
            generador.uniform(-0.5, 0.5, neuronas_por_capa[i + 1])
            for i in range(len(neuronas_por_capa) - 1)
        ]

    # ------------------------------------------------------------------ activacion
    """Sigmoide simetrica:  phi(v) = 2 / (1 + e^(-v)) - 1,  con imagen (-1, +1)."""
    def activacion(self, entrada_neta):
        return 2 / (1 + np.exp(-entrada_neta)) - 1

    """Derivada de la sigmoide simetrica, escrita en funcion de la SALIDA ya calculada:

        phi'(v) = 1/2 * (1 + y) * (1 - y)

    Recibe `y` y no `v` para no tener que guardar las entradas netas ni volver a
    calcular exponenciales: las salidas ya las dejo la pasada hacia adelante.
    """
    def derivada_activacion(self, salida):
        return 0.5 * (1 + salida) * (1 - salida)

    # ------------------------------------------------------------------ hacia adelante

    def propagar(self, entradas):
        """Pasada hacia adelante. Devuelve la lista COMPLETA de activaciones, capa por capa."""
        # La entrada tambien cuenta como activacion: es la salida del nivel 0.
        activaciones = [entradas]

        # zip recorre pesos y umbrales en paralelo: una vuelta por capa de conexiones.
        for pesos_capa, umbral_capa in zip(self.pesos, self.umbrales):
            # activaciones[-1] es lo ultimo que se calculo, o sea la salida de la capa anterior.
            # Producto matricial (@), no elemento a elemento (*).
            #     v_j = suma_i (w_ji * y_i)  +  umbral_j
            entrada_neta = activaciones[-1] @ pesos_capa + umbral_capa

            #     y_j = phi(v_j)
            activaciones.append(self.activacion(entrada_neta))

        # Se devuelven TODAS porque el ajuste de pesos necesita la salida de cada capa.
        return activaciones

    def predecir(self, entradas):
        """Solo la salida final de la red."""
        return self.propagar(entradas)[-1]

    # ------------------------------------------------------------------ un patron

    def entrenar_un_patron(self, entradas, salida_deseada):
        """Un ciclo completo sobre un patron: adelante, error, atras, ajuste."""

        # --- PASO 1: hacia adelante.
        activaciones = self.propagar(entradas)

        # --- PASO 2: el error, y solo en la capa de salida (es la unica con salida deseada).
        #     e_j = d_j - y_j
        error_salida = salida_deseada - activaciones[-1]

        # --- PASO 3: hacia atras, calculando todos los delta.

        # Capa de salida: su delta sale del error de verdad.
        #     delta_j = e_j * phi'(v_j)
        deltas = [error_salida * self.derivada_activacion(activaciones[-1])]

        # Capas ocultas: no tienen error propio. Se les arma el delta con los delta de la
        # capa siguiente, pesados por las conexiones que las unen.
        #     delta_j = ( suma_k delta_k * w_kj ) * phi'(v_j)
        # El range va HACIA ATRAS y se detiene en 1: la capa 0 es la entrada, y la entrada
        # no tiene delta porque no tiene pesos que corregir.
        for capa in range(len(self.pesos) - 1, 0, -1):
            culpa_que_llega = self.pesos[capa] @ deltas[0]      # suma_k delta_k * w_kj
            derivada_local = self.derivada_activacion(activaciones[capa])   # phi'(v_j)
            delta_propagado = derivada_local * culpa_que_llega

            # insert(0, ...) mete al PRINCIPIO: asi la lista queda en orden de capa
            # aunque los delta se calculen del ultimo al primero.
            deltas.insert(0, delta_propagado)

        # --- PASO 4: recien ahora se mueven los pesos, con TODOS los delta ya calculados.
        # Si se mezclara con el paso 3, los delta de las capas de atras saldrian calculados
        # con pesos que ya cambiaron.
        for capa in range(len(self.pesos)):
            #     delta_w_ji = tasa * delta_j * y_i
            # np.outer(a, b)[i, j] = a[i] * b[j]: hace todos los pesos de la capa de una vez.
            correccion = self.tasa_aprendizaje * np.outer(activaciones[capa], deltas[capa])
            self.pesos[capa] = self.pesos[capa] + correccion

            # El umbral cuelga de una entrada fija, asi que no lleva outer:
            #     delta_umbral_j = tasa * delta_j
            self.umbrales[capa] = self.umbrales[capa] + self.tasa_aprendizaje * deltas[capa]

        return error_salida

    # ------------------------------------------------------------------ decision y medicion
    """Convierte las salidas continuas de la red en una clase.

        Con UNA salida: decide el signo (dos clases, +1 y -1).
        Con VARIAS salidas: gana la neurona de mayor salida (winner-takes-all), y devuelve
        su indice. El signo no sirve aca: si dos salidas dan positivo no sabria cual elegir.
    """

    def clasificar(self, entradas):
        salidas = self.predecir(entradas)
        if salidas.shape[1] == 1:
            return np.where(salidas[:, 0] >= 0, 1.0, -1.0)
        return np.argmax(salidas, axis=1)


    """La misma conversion, pero sobre las salidas deseadas, para poder compararlas."""
    def clase_deseada(deseadas):
        if deseadas.ndim == 1:
            return np.where(deseadas >= 0, 1.0, -1.0)
        return np.argmax(deseadas, axis=1)

    """Devuelve (porcentaje de aciertos, cantidad de errores)."""
    def probar(self, entradas, deseadas):
        aciertos = self.clasificar(entradas) == self.clase_deseada(deseadas)
        return 100 * aciertos.mean(), int((~aciertos).sum())

    
    """Copia de los pesos, para guardar la evolucion epoca a epoca."""
    def copiar_estado(self):
        return ([w.copy() for w in self.pesos], [u.copy() for u in self.umbrales])

    # ------------------------------------------------------------------ epocas

    """Entrena por epocas. Una epoca = una pasada por todos los patrones.

        Devuelve (errores_por_epoca, error_cuadratico_por_epoca, historial):
          - errores_por_epoca: cuantos patrones quedan mal clasificados al final de cada epoca
          - error_cuadratico_por_epoca: la suma de xi(n) = 1/2 * suma_j e_j^2 sobre todos los patrones
          - historial: los pesos de cada epoca, solo si guardar_historial=True
    """
    def entrenar(self, entradas, deseadas, maximo_epocas=1000, tolerancia=0.0,
                 guardar_historial=False):

        errores_por_epoca, error_cuadratico_por_epoca, historial = [], [], []

        # Con una sola salida, deseadas viene como vector; el algoritmo la necesita como
        # matriz de una columna para que las formas cierren.
        deseadas_como_matriz = deseadas if deseadas.ndim > 1 else deseadas[:, None]

        for _ in range(maximo_epocas):
            suma_cuadratica = 0.0

            # Entrenamiento PATRON A PATRON: los pesos se ajustan despues de cada patron,
            # no al final de la epoca. Es coherente con el error instantaneo xi(n).
            for entrada, deseada in zip(entradas, deseadas_como_matriz):
                error = self.entrenar_un_patron(entrada, deseada)
                suma_cuadratica += 0.5 * float(np.sum(error ** 2))

            _, cantidad_errores = self.probar(entradas, deseadas)
            errores_por_epoca.append(cantidad_errores)
            error_cuadratico_por_epoca.append(suma_cuadratica)
            if guardar_historial:
                historial.append(self.copiar_estado())

            # Corte anticipado: ningun patron mal clasificado y el error cuadratico medio
            # por debajo de la tolerancia. Con tolerancia=0.0 nunca corta antes de tiempo.
            if cantidad_errores == 0 and suma_cuadratica / len(entradas) < tolerancia:
                break

        return errores_por_epoca, error_cuadratico_por_epoca, historial

## Ejercicio 1

> Implemente el algoritmo de retropropagación para un perceptrón multicapa de forma que
> se pueda elegir libremente la cantidad de capas de la red y de neuronas en cada capa.
> Pruébelo entrenando una red de estructura apropiada para resolver el problema XOR, con
> sus particiones de entrenamiento y prueba correspondientes (datos de la Guía de Trabajos
> Prácticos 1).

In [ ]:
# 1. CARGAR LOS PATRONES ------------------------------------------------------
entradas_xor, deseadas_xor = cargar_patrones(DIRECTORIO_DATASET / "XOR_trn.csv")
entradas_xor_prueba, deseadas_xor_prueba = cargar_patrones(DIRECTORIO_DATASET / "XOR_tst.csv")


# 2. DEFINIR LA CONFIGURACION -------------------------------------------------
CANTIDAD_ENTRADAS = entradas_xor.shape[1]   # 2, lo dicta el dataset
NEURONAS_OCULTAS  = [4]                     # una capa oculta de 4 neuronas
CANTIDAD_SALIDAS  = 1                       # dos clases: alcanza una salida ±1

TASA_APRENDIZAJE = 0.1     # cuanto se mueven los pesos en cada correccion
SEMILLA          = 42      # fija la red inicial: con la misma semilla, el mismo resultado
MAXIMO_EPOCAS    = 200     # cuantas pasadas completas como maximo
TOLERANCIA       = 0.1     # corta antes si no hay errores y el error medio baja de esto

arquitectura_xor = armar_arquitectura(CANTIDAD_ENTRADAS, NEURONAS_OCULTAS, CANTIDAD_SALIDAS)
print("arquitectura:", arquitectura_xor)


# 3. CREAR LA RED -------------------------------------------------------------
red_xor = PerceptronMulticapa(arquitectura_xor, TASA_APRENDIZAJE, SEMILLA)


# 4. ENTRENAR -----------------------------------------------------------------
errores_por_epoca_xor, error_cuadratico_por_epoca_xor, historial_xor = red_xor.entrenar(
    entradas_xor, deseadas_xor,
    maximo_epocas=MAXIMO_EPOCAS, tolerancia=TOLERANCIA, guardar_historial=True,
)


# 5. PROBAR -------------------------------------------------------------------
porcentaje_aciertos, cantidad_errores = red_xor.probar(entradas_xor_prueba, deseadas_xor_prueba)

print(f"épocas usadas:  {len(errores_por_epoca_xor)} de {MAXIMO_EPOCAS}")
print(f"error final:    {error_cuadratico_por_epoca_xor[-1]:.4f}")
print(f"prueba:         {porcentaje_aciertos:.2f} %  "
      f"({cantidad_errores} errores de {len(deseadas_xor_prueba)})")


Cómo se va formando la región de decisión a lo largo del entrenamiento:


In [ ]:
graficar_y_animar_regiones(
    [(entradas_xor, deseadas_xor, historial_xor, "XOR — 2 → 4 → 1")],
    ruta_gif=DIRECTORIO_GRAFICOS / "xor_entrenamiento.gif",
)

casos = []

for ocultas in ([2], [4]):
    fila = []
    for semilla in (42, 0):
        red = PerceptronMulticapa(
            armar_arquitectura(CANTIDAD_ENTRADAS, ocultas, CANTIDAD_SALIDAS),
            TASA_APRENDIZAJE, semilla)
        errores, _, _ = red.entrenar(entradas_xor, deseadas_xor,
                                     maximo_epocas=MAXIMO_EPOCAS, tolerancia=TOLERANCIA)
        convergio = errores[-1] == 0
        porcentaje_aciertos, _ = red.probar(entradas_xor_prueba, deseadas_xor_prueba)

        fila.append({
            "red": red,
            "titulo": f"{ocultas[0]} ocultas · semilla {semilla}",
            "pie": (f"{'converge' if convergio else 'NO converge'} en {len(errores)} épocas"
                    f"  ·  prueba {porcentaje_aciertos:.1f} %"),
            "color": "#1a7f4f" if convergio else "#c0392b",
        })
    casos.append(fila)

graficar_comparacion(casos, entradas_xor, deseadas_xor,
                     ruta_png=DIRECTORIO_GRAFICOS / "xor_comparacion.png")



### Conclusiónes

### Conclusion 1

**La implementación cumple la consigna.** La cantidad de capas y de neuronas por capa se elige
libremente con una lista, y el algoritmo no depende de ella: los pesos se guardan en una lista
de matrices y todos los bucles la recorren. Pasar `[2, 4, 1]` o `[4, 10, 6, 3]` no requiere
tocar una sola línea del algoritmo. La misma clase resuelve, sin modificaciones, los ejercicios
2 y 3 — sólo cambia cómo se la configura y cómo se le dan los datos.

**El XOR se resuelve con una capa oculta.** Con la arquitectura 2 → 4 → 1 la red converge en
dos épocas y clasifica correctamente el 100 % del conjunto de prueba. Es lo esperable desde la
teoría: el XOR necesita una región convexa abierta —una franja entre dos rectas paralelas—, y
eso es exactamente lo que puede formar una capa oculta. Cada neurona oculta aporta una recta y
la de salida las combina.

**Sobre la cantidad de neuronas ocultas.** El mínimo teórico son dos, porque con dos semiplanos
alcanza para armar la franja. La grilla de comparación muestra por qué se usan cuatro: con dos
ocultas y la semilla 42 la red **nunca forma la franja** —el plano queda todo de una sola
clase— y se estanca en 56 %, que es nivel de azar; con la misma arquitectura y otra semilla
converge en dos épocas al 100 %. Con cuatro ocultas converge con las dos inicializaciones.

La diferencia, entonces, **no es que dos neuronas no puedan resolver el XOR**: es que con el
mínimo justo el entrenamiento depende de dónde arranque. Es la distinción entre **existencia**
y **aprendizaje** — la solución con dos existe y se puede construir a mano, pero
back-propagation, partiendo de pesos al azar, no siempre la encuentra. Agregar neuronas por
encima del mínimo **no cambia lo que la red puede representar**: cambia la probabilidad de que
el entrenamiento llegue.

### Conclusión 2


Dos es el **mínimo teórico**: la región que hace falta para el XOR es una franja, o sea la
intersección de dos semiplanos, y para eso alcanzan dos rectas. Pero con el mínimo justo la
convergencia depende mucho de dónde arranquen los pesos, que se inicializan al azar.

La grilla compara las dos arquitecturas contra dos inicializaciones distintas. La única
variable que cambia entre paneles es ésa.

## Ejercicio 2

> Utilice para entrenamiento y prueba los conjuntos de datos `concent_trn.csv` y
> `concent_tst.csv`, que consisten en dos clases distribuidas en forma concéntrica.
> Determine la estructura de una red de tipo perceptrón multicapa que resulte más
> apropiada para resolver este problema. Represente gráficamente, con diferentes colores,
> el resultado de la clasificación realizada por el perceptrón multicapa.

In [ ]:
# 1. CARGAR LOS PATRONES ------------------------------------------------------
entradas_concent, deseadas_concent = cargar_patrones(DIRECTORIO_DATASET / "concent_trn.csv")
entradas_concent_prueba, deseadas_concent_prueba = cargar_patrones(
    DIRECTORIO_DATASET / "concent_tst.csv")


# 2. DEFINIR LA CONFIGURACION -------------------------------------------------
CANTIDAD_ENTRADAS_CONCENT = entradas_concent.shape[1]   # 2, lo dicta el dataset
NEURONAS_OCULTAS_CONCENT  = [8]                         # una capa oculta de 8 neuronas
CANTIDAD_SALIDAS_CONCENT  = 1                           # dos clases: alcanza una salida +-1

TASA_APRENDIZAJE_CONCENT = 0.1     # cuanto se mueven los pesos en cada correccion
SEMILLA_CONCENT          = 0       # fija la red inicial: misma semilla, mismo resultado
MAXIMO_EPOCAS_CONCENT    = 80      # cuantas pasadas completas como maximo
TOLERANCIA_CONCENT       = 0.02    # corta antes si el error medio baja de esto

# Los datos vienen en el cuadrado [0,1]x[0,1]. Se los centra en el origen y se los agranda,
# porque con entradas tan chicas la entrada neta queda diminuta, la sigmoide trabaja en su
# tramo casi recto y la red se comporta como un clasificador lineal.
CENTRO_DE_LOS_DATOS = 0.5    # el centro del cuadrado [0,1]x[0,1]
FACTOR_DE_ESCALA    = 4.0    # cuanto se agranda la nube despues de centrarla

def normalizar_concent(entradas):
    """(x - 0.5) * 4  ->  deja los datos centrados en 0 y repartidos en [-2, 2]."""
    return (entradas - CENTRO_DE_LOS_DATOS) * FACTOR_DE_ESCALA

arquitectura_concent = armar_arquitectura(
    CANTIDAD_ENTRADAS_CONCENT, NEURONAS_OCULTAS_CONCENT, CANTIDAD_SALIDAS_CONCENT)
print("arquitectura:", arquitectura_concent)


# 3. CREAR LA RED -------------------------------------------------------------
red_concent = PerceptronMulticapa(
    arquitectura_concent, TASA_APRENDIZAJE_CONCENT, SEMILLA_CONCENT)


# 4. ENTRENAR -----------------------------------------------------------------
errores_por_epoca_concent, error_cuadratico_por_epoca_concent, _ = red_concent.entrenar(
    normalizar_concent(entradas_concent), deseadas_concent,
    maximo_epocas=MAXIMO_EPOCAS_CONCENT, tolerancia=TOLERANCIA_CONCENT,
)


# 5. PROBAR -------------------------------------------------------------------
porcentaje_aciertos, cantidad_errores = red_concent.probar(
    normalizar_concent(entradas_concent_prueba), deseadas_concent_prueba)

print(f"épocas usadas:  {len(errores_por_epoca_concent)} de {MAXIMO_EPOCAS_CONCENT}")
print(f"error final:    {error_cuadratico_por_epoca_concent[-1]:.4f}")
print(f"prueba:         {porcentaje_aciertos:.2f} %  "
      f"({cantidad_errores} errores de {len(deseadas_concent_prueba)})")


#Graficamos lso reusltados de la clasificación
graficar_clasificacion(
    red_concent,                      # la red ya entrenada
    entradas_concent_prueba,          # se grafica el conjunto de PRUEBA
    deseadas_concent_prueba,          # las clases reales, para el panel de la izquierda
    normalizar_concent,               # el mismo centrado y escalado que se uso al entrenar
    ruta_png=DIRECTORIO_GRAFICOS / "concent_clasificacion.png",
    titulo="Ejercicio 2 — conjunto de prueba",
)


# 6. COMPROBAR QUE 8 ES LA CANTIDAD CORRECTA ----------------------------------
# El razonamiento del paso 1 hace dos predicciones verificables: con 2 neuronas la red NO
# deberia poder cerrar la region, y a partir de 8 no deberia mejorar mas. Se entrena la
# misma red cambiando SOLO ese numero; todo lo demas queda igual.
CANTIDADES_DE_OCULTAS_A_PROBAR = [0,1 ,2, 3, 4, 6, 8, 20]

print("\n--- comprobación de la cantidad de neuronas ocultas ---")
for cantidad_de_ocultas in CANTIDADES_DE_OCULTAS_A_PROBAR:
    red_de_prueba = PerceptronMulticapa(
        armar_arquitectura(CANTIDAD_ENTRADAS_CONCENT, [cantidad_de_ocultas],
                           CANTIDAD_SALIDAS_CONCENT),
        TASA_APRENDIZAJE_CONCENT, SEMILLA_CONCENT)

    red_de_prueba.entrenar(
        normalizar_concent(entradas_concent), deseadas_concent,
        maximo_epocas=MAXIMO_EPOCAS_CONCENT, tolerancia=TOLERANCIA_CONCENT)

    porcentaje, _ = red_de_prueba.probar(
        normalizar_concent(entradas_concent_prueba), deseadas_concent_prueba)

    print(f"{cantidad_de_ocultas:>2} neuronas ocultas  ->  prueba {porcentaje:6.2f} %")

# Referencia: cuanto sacaria una red que contesta siempre la clase mas numerosa. Si alguna
# empata con este numero, esa red no aprendio nada.
proporcion_de_la_clase_mayoritaria = 100 * (deseadas_concent_prueba > 0).mean()
print(f"\nclase mayoritaria (piso, no aprender nada): "
      f"{proporcion_de_la_clase_mayoritaria:.2f} %")


### Extra (fuera de consigna) — la misma región, con una red de base radial

El multicapa llega al 96,4 % y el techo se lo pone la **forma** de la neurona: una sigmoide
sólo sabe partir el plano con una recta, así que el círculo tiene que salir de intersecar
ocho rectas — un octógono. Vale la pena preguntarse qué pasa con una neurona cuya forma
**ya sea** un círculo.

**Qué cambia en la neurona.** La sigmoide calcula $v = \mathbf{w}\cdot\mathbf{x} + b$ y
pregunta *"¿de qué lado de mi recta estás?"*. La gaussiana no tiene recta: tiene un
**centro**, y pregunta *"¿qué tan cerca de mi centro estás?"*

$$\varphi_j(\mathbf{x}) = e^{-\dfrac{\|\mathbf{x}-\mathbf{c}_j\|^2}{2\sigma^2}}$$

Y como $\|\mathbf{x}-\mathbf{c}\| = \text{cte}$ **es una circunferencia**, la neurona radial
trae la región cerrada de fábrica, sin tener que construirla.

**Qué cambia en el entrenamiento.** No hay retropropagación. Son **dos etapas separadas**:
primero **k-medias** ubica los centros mirando sólo la nube de puntos (ni se entera de las
clases), y después **LMS** ajusta los pesos de la capa de salida. Como esa capa es **lineal**,
no hay ningún error que propagar hacia atrás.

Por eso va en una clase aparte y no como una opción de `PerceptronMulticapa`: no es otra
función de activación, es **otro algoritmo**.


In [ ]:
class PerceptronRadial:
    """Red de base radial: capa oculta de gaussianas + capa de salida LINEAL.

    Se diferencia del perceptron multicapa en las dos cosas de fondo:

    1. QUE PREGUNTA CADA NEURONA OCULTA. La sigmoide calcula v = w.x + b y pregunta
       "de que lado de mi recta estas". La gaussiana no tiene recta: tiene un CENTRO,
       y pregunta "que tan cerca de mi centro estas". Como ||x - c|| = constante es una
       circunferencia, la neurona radial ya trae la region cerrada de fabrica.

    2. COMO SE ENTRENA. No hay retropropagacion. Son DOS etapas separadas:
         etapa 1 - k-medias ubica los centros (NO mira las clases: solo la nube de puntos)
         etapa 2 - LMS ajusta los pesos de la capa de salida
       Como la capa de salida es lineal, no hay que propagar ningun error hacia atras.
    """

    def __init__(self, cantidad_de_centros, tasa_aprendizaje=0.1, semilla=None, sigma=None):
        self.cantidad_de_centros = cantidad_de_centros
        self.tasa_aprendizaje = tasa_aprendizaje

        # sigma = el ancho de las gaussianas. Si se deja en None se calcula solo (ver abajo).
        self.sigma = sigma

        self.generador = np.random.default_rng(semilla)

        # Se llenan al entrenar: los centros en la etapa 1, los pesos en la etapa 2.
        self.centros = None
        self.pesos = None
        self.umbral = None

    # ------------------------------------------------------------- etapa 1: k-medias

    def ubicar_centros(self, entradas, maximo_iteraciones=100):
        """k-medias por lotes: reparte los centros donde hay puntos.

        Arranca con centros al azar tomados de los propios patrones, y repite:
          a) cada patron se asigna al centro mas cercano,
          b) cada centro se corre al promedio de los patrones que le tocaron.
        Corta cuando los centros dejan de moverse.
        """
        indices_iniciales = self.generador.choice(
            len(entradas), self.cantidad_de_centros, replace=False)
        centros = entradas[indices_iniciales].copy()

        for _ in range(maximo_iteraciones):
            # a) distancia de cada patron a cada centro, y el mas cercano de todos.
            #    entradas[:, None, :] tiene forma (patrones, 1, dimensiones) y
            #    centros[None, :, :] tiene forma (1, centros, dimensiones): al restarlas
            #    numpy las estira (broadcasting) y sale (patrones, centros, dimensiones).
            distancias_cuadradas = ((entradas[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2)
            centro_asignado = distancias_cuadradas.argmin(axis=1)

            # b) cada centro se muda al promedio de su grupo.
            centros_nuevos = centros.copy()
            for j in range(self.cantidad_de_centros):
                le_tocaron = centro_asignado == j
                if le_tocaron.any():          # un centro sin patrones se deja donde esta
                    centros_nuevos[j] = entradas[le_tocaron].mean(axis=0)

            if np.allclose(centros_nuevos, centros):
                break
            centros = centros_nuevos

        self.centros = centros

        # Ancho de las gaussianas, si no lo fijo el usuario. Regla clasica:
        #     sigma = d_max / raiz(2 * K)
        # con d_max = la mayor distancia entre dos centros. Reparte el ancho para que las
        # gaussianas se solapen lo justo: ni tan angostas que dejen huecos, ni tan anchas
        # que se pisen todas.
        if self.sigma is None:
            distancias_entre_centros = np.sqrt(
                ((centros[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2))
            distancia_maxima = distancias_entre_centros.max()
            self.sigma = distancia_maxima / np.sqrt(2 * self.cantidad_de_centros)

    # ------------------------------------------------------------- activacion radial

    def activacion_radial(self, entradas):
        """Gaussiana N-dimensional, una por centro:

            y_j = exp( - ||x - c_j||^2 / (2 * sigma^2) )

        Vale 1 justo sobre el centro y se cae a 0 al alejarse. Devuelve una matriz de
        (cantidad de patrones, cantidad de centros): la fila de cada patron dice que tan
        cerca esta de cada centro.
        """
        distancias_cuadradas = (
            (entradas[:, None, :] - self.centros[None, :, :]) ** 2).sum(axis=2)
        return np.exp(-distancias_cuadradas / (2 * self.sigma ** 2))

    # ------------------------------------------------------------- hacia adelante

    def predecir(self, entradas):
        """Salida de la red. La capa de salida es LINEAL: no lleva sigmoide.

            y = suma_j (w_j * fi_j(x)) + umbral
        """
        return self.activacion_radial(entradas) @ self.pesos + self.umbral

    # ------------------------------------------------------------- etapa 2: LMS

    def entrenar(self, entradas, deseadas, maximo_epocas=100, tolerancia=0.0,
                 guardar_historial=False):
        """Etapa 1 (una sola vez) + etapa 2 (por epocas). Misma firma que el multicapa."""

        # --- ETAPA 1: ubicar los centros. Se hace UNA vez y no se vuelve a tocar.
        self.ubicar_centros(entradas)

        deseadas_como_matriz = deseadas if deseadas.ndim > 1 else deseadas[:, None]
        cantidad_salidas = deseadas_como_matriz.shape[1]

        self.pesos = self.generador.uniform(
            -0.5, 0.5, (self.cantidad_de_centros, cantidad_salidas))
        self.umbral = self.generador.uniform(-0.5, 0.5, cantidad_salidas)

        # Las salidas de la capa oculta NO cambian durante el entrenamiento, porque los
        # centros ya estan fijos. Se calculan una vez sola y se reusan en cada epoca.
        salidas_ocultas = self.activacion_radial(entradas)

        errores_por_epoca, error_cuadratico_por_epoca, historial = [], [], []

        # --- ETAPA 2: LMS sobre la capa de salida, patron a patron.
        for _ in range(maximo_epocas):
            suma_cuadratica = 0.0

            for salida_oculta, deseada in zip(salidas_ocultas, deseadas_como_matriz):
                #     y = w . fi(x) + umbral        (lineal, sin phi)
                salida = salida_oculta @ self.pesos + self.umbral

                #     e = d - y
                error = deseada - salida

                #     delta_w_j = tasa * e * fi_j     <- LMS: sin phi'(v), porque no hay phi
                self.pesos = self.pesos + self.tasa_aprendizaje * np.outer(salida_oculta, error)
                self.umbral = self.umbral + self.tasa_aprendizaje * error

                suma_cuadratica += 0.5 * float(np.sum(error ** 2))

            _, cantidad_errores = self.probar(entradas, deseadas)
            errores_por_epoca.append(cantidad_errores)
            error_cuadratico_por_epoca.append(suma_cuadratica)
            if guardar_historial:
                historial.append((self.centros.copy(), self.pesos.copy()))

            if cantidad_errores == 0 and suma_cuadratica / len(entradas) < tolerancia:
                break

        return errores_por_epoca, error_cuadratico_por_epoca, historial

    # ------------------------------------------------------------- decision y medicion
    # Identicas a las del multicapa, para que las dos redes se puedan comparar y graficar
    # con las mismas funciones.

    def clasificar(self, entradas):
        salidas = self.predecir(entradas)
        if salidas.shape[1] == 1:
            return np.where(salidas[:, 0] >= 0, 1.0, -1.0)
        return np.argmax(salidas, axis=1)

    @staticmethod
    def clase_deseada(deseadas):
        if deseadas.ndim == 1:
            return np.where(deseadas >= 0, 1.0, -1.0)
        return np.argmax(deseadas, axis=1)

    def probar(self, entradas, deseadas):
        aciertos = self.clasificar(entradas) == self.clase_deseada(deseadas)
        return 100 * aciertos.mean(), int((~aciertos).sum())

**La comparación.** Las dos redes, sobre los mismos datos y el mismo conjunto de prueba.

In [ ]:
# --- las dos redes, mismos datos, mismo conjunto de prueba --------------------
CANTIDADES_A_PROBAR = [2, 4, 8, 20, 40]

print("           multicapa (rectas)      radial (círculos)")
for cantidad in CANTIDADES_A_PROBAR:
    # multicapa: `cantidad` neuronas ocultas sigmoides
    red_mlp = PerceptronMulticapa(
        armar_arquitectura(2, [cantidad], 1), TASA_APRENDIZAJE_CONCENT, SEMILLA_CONCENT)
    red_mlp.entrenar(normalizar_concent(entradas_concent), deseadas_concent,
                     maximo_epocas=MAXIMO_EPOCAS_CONCENT, tolerancia=TOLERANCIA_CONCENT)
    porcentaje_mlp, _ = red_mlp.probar(
        normalizar_concent(entradas_concent_prueba), deseadas_concent_prueba)

    # radial: `cantidad` centros de gaussiana
    red_rbf = PerceptronRadial(cantidad, tasa_aprendizaje=0.05, semilla=SEMILLA_CONCENT)
    red_rbf.entrenar(normalizar_concent(entradas_concent), deseadas_concent,
                     maximo_epocas=60, tolerancia=TOLERANCIA_CONCENT)
    porcentaje_rbf, _ = red_rbf.probar(
        normalizar_concent(entradas_concent_prueba), deseadas_concent_prueba)

    print(f"{cantidad:>3} neuronas:      {porcentaje_mlp:6.2f} %            {porcentaje_rbf:6.2f} %")


# --- la radial NO necesita que se normalicen las entradas ---------------------
# La gaussiana mide distancias y sigma se calcula a partir de los propios centros,
# asi que si los datos se agrandan, sigma se agranda junto con ellos.
print("\nradial de 20 centros:")
for etiqueta, transformacion in [("con datos normalizados", normalizar_concent),
                                 ("con datos crudos      ", lambda entradas: entradas)]:
    red = PerceptronRadial(20, tasa_aprendizaje=0.05, semilla=SEMILLA_CONCENT)
    red.entrenar(transformacion(entradas_concent), deseadas_concent,
                 maximo_epocas=60, tolerancia=TOLERANCIA_CONCENT)
    print(f"  {etiqueta} -> {red.probar(transformacion(entradas_concent_prueba), deseadas_concent_prueba)[0]:6.2f} %")


# --- el grafico de la radial, para comparar con el del multicapa --------------
red_radial_concent = PerceptronRadial(20, tasa_aprendizaje=0.05, semilla=SEMILLA_CONCENT)
red_radial_concent.entrenar(normalizar_concent(entradas_concent), deseadas_concent,
                            maximo_epocas=60, tolerancia=TOLERANCIA_CONCENT)

graficar_clasificacion(
    red_radial_concent, entradas_concent_prueba, deseadas_concent_prueba, normalizar_concent,
    ruta_png=DIRECTORIO_GRAFICOS / "concent_radial.png",
    titulo="Ejercicio 2 — base radial, 20 centros",
)

### Conclusiones

### Conclusión 1 — la estructura es 2 → 8 → 1

Se determinó razonando la forma **antes** de probar números:

1. **Entradas y salidas.** Los datos tienen dos coordenadas → capa de entrada de **2**. Hay
   dos clases → alcanza **una** salida, y el signo de $y$ dice cuál es.
2. **Capas ocultas: una.** La clase de adentro está encerrada, así que la región tiene que ser
   **cerrada**. Un recinto cerrado y aproximadamente circular es **convexo**, y las regiones
   convexas las da **una sola capa oculta**. No hay concavidades ni agujeros que justifiquen
   una segunda.
3. **Neuronas en esa capa: ocho.** Cada neurona oculta aporta **una recta** y la capa de salida
   las interseca: **N neuronas dan un polígono de hasta N lados**. Con 2 no se puede cerrar
   nada; con 8 el polígono ya aproxima bien la circunferencia.

**Los números confirman los tres pasos:**

| neuronas ocultas | prueba |
|---|---|
| 2 | **63,10 %** — el piso, la clase mayoritaria |
| 4 | 94,40 % |
| **8** | **96,40 %** |
| 20 | 95,40 % — no mejora |

Con 2 la red **no aprende nada**: 63,10 % es exactamente la proporción de la clase más
numerosa. No es mala suerte, es que dos rectas no cierran una región. Y de 8 en adelante deja
de mejorar: el techo ya no lo pone la arquitectura sino la **geometría**. Los 36 errores están
todos sobre el borde, que es la diferencia entre un **polígono de ocho lados y un círculo**.

---

### Conclusión 2 — hay que normalizar las entradas

Sin normalizar, la misma red de 8 ocultas **también se queda en 63,10 %**. La razón es que la
red se vuelve una recta sin que se note.

**La sigmoide tiene un tramo recto.** Vista de lejos $\varphi(v) = \frac{2}{1+e^{-v}}-1$ es una
S, pero entre $v=-1$ y $v=1$ es **indistinguible de la recta $v/2$**. Ahí no dobla nada.

![La sigmoide de cerca y de lejos](Graficos/sigmoide_zoom.png)

**Y una red que trabaja en ese tramo ES una recta.** Si $\varphi(v)\approx v/2$, cada neurona
oculta queda siendo una función **lineal** de $\mathbf{x}$, y la capa de salida combina
linealmente funciones lineales: **el resultado sigue siendo lineal**. Da lo mismo apilar ocho
neuronas o tres capas — componer lineales da lineal.

Es el argumento de la teoría dado vuelta: **la no linealidad de $\varphi$ es lo único que hace
que valga la pena tener capas ocultas**. En el tramo recto esa no linealidad está apagada, y
una recta sobre dos clases concéntricas no puede hacer más que votar la clase mayoritaria.

**Los datos crudos caen justo ahí.** Vienen en el cuadrado $[0,1]^2$ y los pesos iniciales son
chicos, así que $v = \mathbf{w}\cdot\mathbf{x}+b$ queda chico:

| entradas | rango de $v$ |
|---|---|
| crudas | $-0{,}36$ a $+0{,}76$ → **todo en el tramo recto** |
| $(x-0{,}5)\cdot 4$ | $-1{,}09$ a $+1{,}84$ → ya asoma donde **dobla** |

![Dónde caen los datos sobre la sigmoide](Graficos/sigmoide_datos.png)

**Normalizar no cambia el problema, mueve los datos.** Centrar (restar 0,5) los lleva al
origen, donde la sigmoide es simétrica; escalar (por 4) multiplica $v$ por 4, porque $v$ es
lineal en $\mathbf{x}$. El círculo sigue siendo el mismo círculo.

| entradas | prueba |
|---|---|
| crudas | 63,10 % |
| sólo centradas | 63,10 % |
| sólo escaladas | 93,40 % |
| **centradas y escaladas** | **96,40 %** |

**El que pesa es el escalado**, que es el que saca a la sigmoide del tramo recto. Centrar suma
tres puntos porque con todas las entradas positivas las correcciones empujan siempre para el
mismo lado y el descenso zigzaguea.

*Matiz honesto:* esto describe el estado **inicial**. Los pesos crecen durante el
entrenamiento, así que la red podría salir sola del tramo recto. Lo que da la normalización no
es que sin ella sea imposible aprender, sino **arrancar donde el gradiente tiene de qué
agarrarse** en vez de gastar las 80 épocas haciendo crecer los pesos.

---

### Extra (fuera de consigna) — comparación con la red de base radial

| neuronas | multicapa (rectas) | radial (círculos) |
|---|---|---|
| 2 | **63,10 %** | 88,10 % |
| 4 | 94,40 % | 92,20 % |
| 8 | **96,40 %** | 95,10 % |
| 20 | 95,40 % | 97,50 % |
| 40 | 94,10 % | **98,20 %** |

**Con 2 neuronas la diferencia es total.** Dos rectas no pueden encerrar una región, así que el
multicapa se queda en el piso; dos gaussianas sí, porque **cada una ya es una región cerrada**.
Misma cantidad de neuronas, resolviendo o no según la forma que cada una sabe dibujar.

**Las curvas se cruzan.** Hasta 8 gana el multicapa; de 20 en adelante gana la radial. El
multicapa toca techo y después empeora — rectas de más no agregan lados útiles y sí agregan
mínimos locales. La radial sigue subiendo, porque un centro de más es cobertura de más.

**La radial no necesita normalizar:** 97,50 % con datos normalizados y 97,50 % con datos
crudos. La gaussiana mide distancias y $\sigma$ se calcula de la separación entre los propios
centros, así que si los datos se agrandan, $\sigma$ se agranda con ellos. **No hay tramo recto
del que haya que sacarla.**

**No es que una sea mejor.** `concent` está hecho de círculos y la radial trae círculos. Si el
problema fuese separar con una diagonal, el perceptrón lo haría con **una** neurona y la radial
necesitaría muchísimas gaussianas. Se elige **la neurona cuya forma se parece al problema**.


## Ejercicio 3

> Iris es el género de una planta herbácea con flores que se utilizan en decoración.
> Dentro de este género existen muy diversas especies, entre las que se han estudiado:
> *Iris setosa*, *Iris versicolor* e *Iris virginica*. Estas tres especies pueden
> distinguirse según las dimensiones de sus pétalos y sépalos. Un grupo de investigadores
> ha recopilado la información correspondiente a las longitudes y anchos de los pétalos y
> sépalos de 50 plantas de cada especie. En el archivo `iris81_trn.csv` se encuentra el
> conjunto de entrenamiento, y en `iris81_tst.csv` el de prueba, generado a partir de estas
> mediciones (en cm), junto con un código binario que indica la clase de cada muestra
> (especie) reconocida por el grupo de investigadores ($[-1,-1,1]$ = setosa,
> $[-1,1,-1]$ = versicolor, $[1,-1,-1]$ = virginica).
>
> Determine la estructura óptima de un perceptrón multicapa para resolver este problema.
> Explore cómo varía el desempeño al usar distintas tasas de aprendizaje, y para cada caso
> grafique las curvas de error cuadrático total y error de clasificación en función de las
> épocas de entrenamiento.

In [ ]:
from IPython.display import display   # para mostrar mas de una tabla en la misma celda

# 1. CARGAR LOS PATRONES ------------------------------------------------------
# n_salidas=3 porque la especie viene codificada en TRES columnas, no en una.
entradas_iris, deseadas_iris = cargar_patrones(
    DIRECTORIO_DATASET / "iris81_trn.csv", n_salidas=3)
entradas_iris_prueba, deseadas_iris_prueba = cargar_patrones(
    DIRECTORIO_DATASET / "iris81_tst.csv", n_salidas=3)

print("patrones de entrenamiento:", len(entradas_iris))
print("patrones de prueba:       ", len(entradas_iris_prueba))


# 2. DEFINIR LA CONFIGURACION -------------------------------------------------
CANTIDAD_ENTRADAS_IRIS = entradas_iris.shape[1]        # 4: largo y ancho de petalo y sepalo
CANTIDAD_SALIDAS_IRIS  = deseadas_iris.shape[1]        # 3: una neurona por especie

SEMILLA_IRIS       = 0
MAXIMO_EPOCAS_IRIS = 150
TOLERANCIA_IRIS    = 0.0        # 0.0 = nunca corta antes: se usan siempre las 150 epocas

# Las cuatro medidas estan en centimetros con rangos muy distintos: el largo del petalo tiene
# un desvio casi 4 veces mayor que el ancho del sepalo, y domina el producto interno. Se las
# lleva a media 0 y desvio 1 para que las cuatro pesen igual.
media_iris  = entradas_iris.mean(axis=0)
desvio_iris = entradas_iris.std(axis=0)

def normalizar_iris(entradas):
    """(x - media) / desvio, columna por columna."""
    return (entradas - media_iris) / desvio_iris


# 3. DETERMINAR LA ESTRUCTURA -------------------------------------------------
# Las tres especies se separan bien con hiperplanos, asi que en principio no haria falta capa
# oculta: las 3 neuronas de salida ya trazan 3 hiperplanos en R^4. Se comprueba comparando
# contra dos redes con capa oculta, y se cuenta cuantos parametros usa cada una.
print("\n--- estructura ---")
for neuronas_ocultas in ([], [4], [10]):
    red = PerceptronMulticapa(
        armar_arquitectura(CANTIDAD_ENTRADAS_IRIS, neuronas_ocultas, CANTIDAD_SALIDAS_IRIS),
        0.05, SEMILLA_IRIS)
    red.entrenar(normalizar_iris(entradas_iris), deseadas_iris,
                 maximo_epocas=MAXIMO_EPOCAS_IRIS, tolerancia=TOLERANCIA_IRIS)
    porcentaje, _ = red.probar(normalizar_iris(entradas_iris_prueba), deseadas_iris_prueba)
    cantidad_parametros = sum(w.size for w in red.pesos) + sum(u.size for u in red.umbrales)
    print(f"ocultas {str(neuronas_ocultas):>5}: {cantidad_parametros:>3} parámetros"
          f" -> prueba {porcentaje:6.2f} %")

# Las tres clasifican igual, asi que se elige la mas chica: 4 -> 3, sin capa oculta.
NEURONAS_OCULTAS_IRIS = []


# 4. EXPLORAR LAS TASAS DE APRENDIZAJE ----------------------------------------
TASAS_A_PROBAR = [0.01, 0.05, 0.1, 0.3, 0.5, 0.9, 1.5]

curvas_por_tasa = {}      # tasa -> (error cuadratico por epoca, errores por epoca)
filas_de_la_tabla = []

for tasa in TASAS_A_PROBAR:
    red = PerceptronMulticapa(
        armar_arquitectura(CANTIDAD_ENTRADAS_IRIS, NEURONAS_OCULTAS_IRIS, CANTIDAD_SALIDAS_IRIS),
        tasa, SEMILLA_IRIS)
    errores_por_epoca, error_cuadratico_por_epoca, _ = red.entrenar(
        normalizar_iris(entradas_iris), deseadas_iris,
        maximo_epocas=MAXIMO_EPOCAS_IRIS, tolerancia=TOLERANCIA_IRIS)

    curvas_por_tasa[tasa] = (error_cuadratico_por_epoca, errores_por_epoca)
    filas_de_la_tabla.append({
        "tasa μ": tasa,
        "ξ final": round(error_cuadratico_por_epoca[-1], 2),
        # desvio de xi en las ultimas 10 epocas: mide si la curva se asento o sigue oscilando
        "desvío de ξ (últimas 10)": round(float(np.std(error_cuadratico_por_epoca[-10:])), 4),
        "error de clasificación [%]": round(100 * errores_por_epoca[-1] / len(entradas_iris), 2),
        "prueba [%]": round(red.probar(normalizar_iris(entradas_iris_prueba),
                                       deseadas_iris_prueba)[0], 2),
    })

print("\n--- tasas de aprendizaje ---")
display(pd.DataFrame(filas_de_la_tabla))

# Las curvas que pide la consigna: error cuadratico total y error de clasificacion vs epoca.
graficar_curvas_por_tasa(curvas_por_tasa, cantidad_patrones=len(entradas_iris),
                         ruta_png=DIRECTORIO_GRAFICOS / "iris_curvas.png")


# 5. ENTRENAR Y PROBAR LA RED ELEGIDA -----------------------------------------
TASA_APRENDIZAJE_IRIS = 0.05        # la mejor del barrido de arriba

arquitectura_iris = armar_arquitectura(
    CANTIDAD_ENTRADAS_IRIS, NEURONAS_OCULTAS_IRIS, CANTIDAD_SALIDAS_IRIS)
red_iris = PerceptronMulticapa(arquitectura_iris, TASA_APRENDIZAJE_IRIS, SEMILLA_IRIS)
print("\n--- red elegida ---")
print("arquitectura:", arquitectura_iris)

red_iris.entrenar(normalizar_iris(entradas_iris), deseadas_iris,
                  maximo_epocas=MAXIMO_EPOCAS_IRIS, tolerancia=TOLERANCIA_IRIS)

porcentaje_aciertos, cantidad_errores = red_iris.probar(
    normalizar_iris(entradas_iris_prueba), deseadas_iris_prueba)
print(f"prueba: {porcentaje_aciertos:.2f} %  "
      f"({cantidad_errores} errores de {len(entradas_iris_prueba)})")

# Matriz de confusion: que especie predijo la red contra cual era en realidad.
# El indice sale de argmax sobre [-1,-1,1]=setosa, [-1,1,-1]=versicolor, [1,-1,-1]=virginica.
especies = {2: "setosa", 1: "versicolor", 0: "virginica"}
predichas = red_iris.clasificar(normalizar_iris(entradas_iris_prueba))
reales    = red_iris.clase_deseada(deseadas_iris_prueba)

display(pd.crosstab(pd.Series([especies[c] for c in reales], name="real"),
                    pd.Series([especies[c] for c in predichas], name="predicha")))

### Conclusiones

### Conclusión 1 — la estructura óptima es 4 → 3, sin capa oculta

| ocultas | parámetros | prueba |
|---|---|---|
| **ninguna** | **15** | **100 %** |
| [4] | 35 | 100 % |
| [10] | 83 | 100 % |

Las tres clasifican igual, así que se elige **la de menos parámetros**. Y el resultado tiene
sentido teórico: **Iris es casi linealmente separable**, y las tres neuronas de salida ya
trazan tres hiperplanos en $\mathbb{R}^4$, que es todo lo que hace falta. Agregar capas ocultas
no mejora nada porque **no hay ninguna región curva ni cerrada que formar** — al revés que en
`concent`, donde la capa oculta era lo único que permitía cerrar el círculo.

---

### Conclusión 2 — la tasa de aprendizaje tiene tres zonas

Las curvas lo muestran mejor que la tabla:

- **μ chico (0,01):** aprende, pero **lento**. A las 150 épocas el error todavía está bajando y
  el error de clasificación se queda en 4,5 %, contra el 0,9 % de las tasas intermedias.
- **μ intermedio (0,05 y 0,1):** lo mejor. Baja rápido, queda estable y clasifica el **100 %**
  del conjunto de prueba.
- **μ grande (0,3 en adelante):** **empeora**, y la prueba cae hasta el 78 %. Con μ = 1,5 la
  curva de error cuadrático **oscila** en vez de bajar: el desvío de $\xi$ en las últimas diez
  épocas es 0,48 contra 0,009 con μ = 0,1, **cincuenta veces más**. No es ruido numérico — el
  paso se volvió tan grande que el descenso **se pasa del mínimo y rebota**.

Es el compromiso del gradiente descendente de la teoría: μ chico converge seguro pero lento,
μ grande avanza rápido hasta que se pasa. El umbral teórico es $\mu < 2/\lambda_{max}$.

---

### Conclusión 3 — cómo leer el 100 %

Con `4 → 3` y μ = 0,05 la red clasifica bien **los 37 patrones de prueba**, y la matriz de
confusión no tiene nada fuera de la diagonal.

Pero conviene relativizarlo: el conjunto de prueba tiene **patrones repetidos** — 8 valores
aparecen dos o tres veces entre los 37 — así que cada patrón vale 2,7 % y uno duplicado vale
5,4 %. Un salto de 94,6 % a 100 % puede ser **un solo punto contado dos veces**, no dos errores
distintos.

El punto crítico es $(6{,}1,\ 3{,}0,\ 4{,}9,\ 1{,}8)$, en pleno solapamiento entre *versicolor*
y *virginica*: mirando sólo esas cuatro medidas es **genuinamente ambiguo**, y ninguna red lo
va a resolver siempre.
